# 11.8 Shuffle Hash Joins

## Learning Objectives

By the end of this lesson you will be able to:

- Understand what a Shuffle Hash Join is
- Learn how Shuffle Hash Join works internally
- Understand the execution flow
- Learn how hash tables are used
- Compare Shuffle Hash Join with Sort Merge Join
- Understand performance trade-offs

> **Core Rule:** Shuffle Hash Join is an alternative join strategy Spark may use when joining large datasets to avoid expensive sorting operations.

## Setup: Initialize Spark & Mock Data

Let's start our local Spark Session and generate two datasets. We will use these to demonstrate how Spark can bypass sorting and use Hash Tables instead.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Shuffle_Hash_Joins_Deep_Dive") \
    .master("local[*]") \
    .getOrCreate()

# Create Dataset A (e.g., 100,000 rows)
df_a = spark.range(1, 100000).withColumnRenamed("id", "user_id") \
            .withColumn("score", F.rand() * 100)

# Create Dataset B (e.g., 30,000 rows)
df_b = spark.range(1, 30000).withColumnRenamed("id", "user_id") \
            .withColumn("status", F.lit("Active"))

print("Spark Session Initialized and Mock Data Created!")

# Recap: Previous Join Strategies

So far we have studied:
1. **Broadcast Join:** Best for small tables (fastest, no shuffle).
2. **Sort Merge Join:** Best for large distributed datasets (heavy shuffle, requires sorting).

Now we will learn a third strategy: **Shuffle Hash Join**.

### Why Does Spark Need Another Join Strategy?
You might wonder: *"If Sort Merge Join already works for large datasets, why do we need Shuffle Hash Join?"*

The answer is: **Sorting can be extremely expensive.** 
If Spark can avoid sorting and use hashing instead, execution may become significantly faster in some situations.

# What is a Shuffle Hash Join?

Shuffle Hash Join is a join strategy that:
1. Shuffles data by join key
2. Builds hash tables
3. Uses hash lookups to find matches

Unlike Sort Merge Join, it uses hashing instead of sorting.

<h3>Shuffle Hash Join Overview</h3>
<img src="../assets/Module_11/11_08_01.png" alt="Shuffle Hash Join Overview" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark Shuffle Hash Join overview showing two datasets entering shuffle phase, hash table creation phase and matching phase. Educational architecture diagram with arrows and labeled stages.</i></p>

# The Basic Idea Behind Hashing

A hash function converts a value into a specific location (a hash key/bucket).

**Example:**
`Customer ID = 101` → `Hash Function` → `Hash Bucket = 5`

Records with the same join key generate the same hash location. This makes matching virtually instantaneous (O(1) time complexity) compared to scanning.

**Real-World Analogy:**
Imagine a library. Instead of walking down every aisle scanning every shelf sequentially (Sorting/Merging), books are registered in a digital directory system (Hash Table). You directly go to the exact section you need.

<h3>Hash Table Concept</h3>
<img src="../assets/Module_11/11_08_02.png" alt="Hash Table Concept" width="75%">
<p><i><b>Image Prompt:</b> Educational hash table visualization showing customer IDs mapped into hash buckets. Beginner-friendly infographic explaining hashing concepts.</i></p>

# High-Level Execution Flow

Shuffle Hash Join consists of three phases:

- **Phase 1:** Shuffle
- **Phase 2:** Build Hash Table
- **Phase 3:** Hash-Based Matching

Let's examine each phase individually.

## Phase 1: Shuffle

Like Sort Merge Join, Shuffle Hash Join begins with data redistribution.

Records with the same join key are moved across the network into the same partition on the same executor. This ensures matching records eventually reach the same physical location.

<h3>Shuffle Phase</h3>
<img src="../assets/Module_11/11_08_03.png" alt="Shuffle Phase" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark shuffle process for Shuffle Hash Join. Matching keys moving across executors into common partitions. Educational cluster visualization.</i></p>

## Phase 2: Build Hash Table

After shuffling, Spark selects one side of the join (usually the smaller of the two partitions).

An in-memory hash table is created using the join key.

**Why Build a Hash Table?**
Without it, Spark would have to repeatedly loop over records. With a hash table, Spark can store records in memory for immediate, direct lookup.

<h3>Hash Table Creation</h3>
<img src="../assets/Module_11/11_08_04.png" alt="Hash Table Creation" width="75%">
<p><i><b>Image Prompt:</b> Spark executor building an in-memory hash table from shuffled records. Hash buckets storing records by join key. Educational architecture diagram.</i></p>

## Phase 3: Hash-Based Matching

The second dataset is scanned.

For every join key, Spark performs a hash lookup against the table built in Phase 2.

**Example:**
- **Hash Table:** `101 → Customer A`, `102 → Customer B`
- **Incoming Order:** `CustomerID = 102`
- **Result:** Spark immediately finds `Customer B` using hash lookup.

<h3>Hash-Based Matching</h3>
<img src="../assets/Module_11/11_08_05.png" alt="Hash Matching" width="75%">
<p><i><b>Image Prompt:</b> Hash lookup process showing incoming records matching entries in an existing hash table. Educational infographic demonstrating efficient join matching.</i></p>

# Complete Execution Flow

This flow completely replaces the *sorting* phase used in Sort Merge Join.

```text
Dataset A                       Dataset B
    ↓                               ↓
 Shuffle                         Shuffle
    ↓                               ↓
Build Hash Table               Hash Lookup
    ↓                               ↓
    +-------- Join Result ----------+
```

<h3>Complete Shuffle Hash Join Flow</h3>
<img src="../assets/Module_11/11_08_06.png" alt="Complete Flow" width="75%">
<p><i><b>Image Prompt:</b> End-to-end Shuffle Hash Join execution flow showing shuffle, hash table creation and hash lookup stages producing final joined output.</i></p>

## Forcing a Shuffle Hash Join in PySpark

By default, modern Spark prefers Sort Merge Joins because they are safer for memory. To see a Shuffle Hash Join, we must manually configure Spark's optimizer.

In [ ]:
# 1. Turn OFF Auto Broadcast Join (so it doesn't default to BroadcastHashJoin)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# 2. Turn OFF Spark's preference for Sort Merge Joins
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

print("Configurations updated. Executing Join...")

# Perform the join
shj_df = df_a.join(df_b, "user_id")

# Trigger an action
print(f"Join completed! Result row count: {shj_df.count():,}")

# Trade-Offs: Shuffle Hash Join vs Sort Merge Join

**Sort Merge Join:** `Shuffle → Sort → Merge`
**Shuffle Hash Join:** `Shuffle → Build Hash Table → Match`

### Trade-Off 1: Less Sorting (Advantage)
No large sorting phase means reduced CPU work and potentially much faster execution. This is the primary strength of Shuffle Hash Join.

### Trade-Off 2: Higher Memory Usage (Disadvantage)
Hash tables must be stored entirely in memory. If the partition is too large, it can cause memory pressure, executor instability, and Out-of-Memory (OOM) errors.

<h3>Memory Usage in Shuffle Hash Join</h3>
<img src="../assets/Module_11/11_08_07.png" alt="Memory Usage" width="75%">
<p><i><b>Image Prompt:</b> Spark executors storing large in-memory hash tables. Memory consumption highlighted as a critical resource. Educational performance diagram.</i></p>

### Trade-Off 3: It Still Requires Shuffle!

A common misconception: *"Shuffle Hash Join eliminates Shuffle."*

**This is FALSE.** 

Shuffle Hash Join still requires data redistribution, network transfer, and partition alignment. The heavy network cost of Shuffle still exists.

<h3>Shuffle Still Exists</h3>
<img src="../assets/Module_11/11_08_08.png" alt="Shuffle Exists" width="75%">
<p><i><b>Image Prompt:</b> Comparison showing Shuffle Hash Join still requiring network shuffle before hash table creation. Educational infographic correcting common misconceptions.</i></p>

# Shuffle Hash Join in Execution Plans

Physical Plans reveal the join strategy. Let's look at the plan for the join we just executed to verify that Spark actually used our requested strategy.

In [ ]:
print("Physical Plan for our Shuffle Hash Join:")
print("-"*50)
shj_df.explain()
print("-"*50)
print("Notice the 'ShuffledHashJoin' operator. There is NO 'Sort' operator!")

# Reset configurations back to best practice defaults
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

# Example Physical Plan

In the output above, you see:

ShuffledHashJoin
      ↓
Exchange

This indicates:
- Data redistribution occurred (`Exchange`)
- Hash-based matching is being used (`ShuffledHashJoin`)

<h3>ShuffledHashJoin Operator</h3>
<img src="../assets/Module_11/11_08_09.png" alt="ShuffledHashJoin Operator" width="75%">
<p><i><b>Image Prompt:</b> Spark Physical Plan highlighting ShuffledHashJoin and Exchange operators. Educational execution plan visualization showing hash-based join processing.</i></p>

# Comparing All Join Strategies

| Feature | Broadcast Join | Sort Merge Join | Shuffle Hash Join |
|----------|----------|----------|----------|
| **Shuffle** | Minimal | Yes | Yes |
| **Sorting** | No | Yes | No |
| **Hash Table** | No | No | Yes |
| **Memory Usage**| Moderate | Moderate | High |
| **Large Dataset**| Limited | Excellent | Good |

Each strategy has different strengths and weaknesses.

# Key Takeaways & Real-World Perspective

Most Spark workloads use **Broadcast Join** and **Sort Merge Join** more frequently. 
Shuffle Hash Join appears less often because of its memory risk, but understanding it helps explain Spark's optimizer decisions and execution plans.

- Shuffle Hash Join uses hashing instead of sorting.
- Execution consists of: Shuffle → Hash Table Creation → Hash-Based Matching.
- It can outperform Sort Merge Join by avoiding CPU-heavy sorting.
- It requires sufficient memory, otherwise executors crash.
- **Shuffle cost still exists.**
- Physical Plans reveal `ShuffledHashJoin` operators.

---

# Next Lesson: 11.9 Data Skewness

In the next lesson we will learn:
- What is Data Skew?
- Detecting Skew
- Real-World Examples

Data Skew is the #1 most common performance problem encountered during any Shuffle operation!